##Import statements, configure notebook presentation

In [ ]:
# pip install geojson

In [ ]:
import json
import pandas as pd
import datetime
import plotly.express as px

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

# Load, clean and format the data

In [ ]:
data = pd.read_csv('otodom_data.csv')

In [ ]:
data = data[~data["District"].isin(['Jabłonna', 'Łomianki', 'Lesznowola', 'Michałowice', 'Piastów', 'StareBabice', 'Wiązowna', 'Raszyn', 'Konstancin-Jeziorna', 'Marki', 'Piaseczno', 'Ząbki'])]

In [ ]:
data.Available_from = pd.to_datetime(data.Available_from)

In [ ]:
data = data.drop_duplicates(subset='ID', ignore_index=True)

In [ ]:
id_to_drop = [67178687, 67150833, 67052619, 67165116, 67137679]
data = data[~data['ID'].isin(id_to_drop)]

In [ ]:
data_without_price = data[data.Price_PLN == 'Zapytaj o cenę']
data_with_price = data[~data['ID'].isin(data_without_price['ID'])]

In [ ]:
data_with_price["Price_PLN"] = (
    data_with_price["Price_PLN"]
    .astype(str)
    .str.replace(r"[ zł]", "", regex=True)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

/tmp/ipython-input-3564188060.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_with_price["Price_PLN"] = (


# Preliminary Data Exploration

In [ ]:
print('Shape of:')
print('- all data:', data.shape)
print('- data without price:', data_without_price.shape)
print('- data with price:', data_with_price.shape)

Shape of:
- all data: (14917, 21)
- data without price: (2305, 21)
- data with price: (12612, 21)


In [ ]:
data.columns

Index(['URL', 'ID', 'Title', 'Price_PLN', 'Square_meter', 'Price_sq_meter',
       'Rooms', 'Street', 'Subdistrict', 'District', 'City', 'Province',
       'Heating', 'Floor', 'Rent_PLN', 'Condition', 'Market', 'Ownership',
       'Available_from', 'Advertiser_type', 'Seller'],
      dtype='object')

In [ ]:
data_with_price.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12612 entries, 0 to 14921
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   URL              12612 non-null  object        
 1   ID               12612 non-null  int64         
 2   Title            12612 non-null  object        
 3   Price_PLN        12612 non-null  float64       
 4   Square_meter     12612 non-null  float64       
 5   Price_sq_meter   12457 non-null  float64       
 6   Rooms            12612 non-null  int64         
 7   Street           10084 non-null  object        
 8   Subdistrict      12595 non-null  object        
 9   District         12595 non-null  object        
 10  City             12595 non-null  object        
 11  Province         12595 non-null  object        
 12  Heating          9783 non-null   object        
 13  Floor            12390 non-null  object        
 14  Rent_PLN         9356 non-null   float64   

In [ ]:
data_with_price.describe()

,ID,Price_PLN,Square_meter,Price_sq_meter,Rooms,Rent_PLN,Available_from
count,"12,612.00","12,612.00","12,612.00","12,457.00","12,612.00","9,356.00",6647
mean,"66,868,519.65","1,169,132.18",60.40,"18,711.95",2.61,968.50,2025-08-07 10:15:15.300135680
min,"33,133,193.00","160,000.00",14.50,"5,587.49",1.00,0.00,1980-01-01 00:00:00
25%,"66,809,406.50","682,875.00",41.00,"14,687.50",2.00,500.00,2025-06-19 00:00:00
50%,"67,058,487.50","878,249.00",53.00,"17,385.84",3.00,731.50,2025-08-15 00:00:00
75%,"67,158,537.50","1,260,693.25",69.01,"20,836.13",3.00,985.50,2025-09-08 00:00:00
max,"67,221,667.00","25,000,000.00",456.00,"92,470.92",10.00,"750,000.00",2028-12-31 00:00:00
std,"779,217.57","1,049,392.69",31.44,"6,590.88",1.00,"11,173.18",NaN


In [ ]:
fig = px.pie(data,
             names='Market',
             color='Market',
             hole=.3,
             color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                 'pierwotny':px.colors.sequential.haline[10]},
             labels={'Market':'Rynek'})
fig.update_traces(textfont=dict(size=14),
                  textposition='inside',
                  textinfo='percent+label')
fig.update_layout(showlegend=False)
fig.show()
fig.write_html('pie.html')

#Price / Price per Square distribution analysis


### Listings without a specified price

In [ ]:
adv_without_price = int(data.Price_PLN[data.Price_PLN == 'Zapytaj o cenę'].count())
all_adv = int(data.Price_PLN.count())
percentage_without_price = round((adv_without_price / all_adv) * 100, 2)
print(f'Listings without a specified price: {percentage_without_price} %.')

Listings without a specified price: 15.45 %.


### Number of Listings by Price, separated by Primary and Secondary Market



In [ ]:
fig = px.histogram(data_with_price,
                   x='Price_PLN',
                   color='Market',
                   hover_data=data_with_price.columns,
                   template='plotly_white',
                   range_y=(0, 1050),
                   range_x=(0, 5000000),
                   color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                       'pierwotny':px.colors.sequential.haline[10]},
                   labels={'Market':'Rynek',
                           'Price_PLN':'Cena',
                           'count':'Ilość ogłoszeń'})
fig.update_layout(xaxis_title='<b>Cena nieruchomości</b>',
                  yaxis_title='<b>Liczba ogłoszeń</b>')

# title='<b>Ilość ogłoszeń w zależnośći od ceny</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>'
fig.show()
fig.write_html('hist_liczba_cena.html')

### Number of Listings by Price per m², separated by Primary and Secondary Market

In [ ]:
fig = px.histogram(data_with_price, x='Price_sq_meter',
                   color='Market',
                   hover_data=data.columns,
                   template='plotly_white',
                   range_y=(0, 750),
                   range_x=(0, 50010),
                   color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                       'pierwotny':px.colors.sequential.haline[10]},
                   labels={'Market':'Rynek',
                           'Price_sq_meter':'Cena za m2'})
fig.update_layout(xaxis_title='<b>Cena za m<sup>2</sup></b>',
                  yaxis_title='<b>Liczba ogłoszeń</b>',)
# title='<b>Ilość ogłoszeń w zależnośći od ceny za m2</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>',
fig.show()
fig.write_html('hist_liczba_cena_m2.html')

In [ ]:
data.Square_meter.describe()

,Square_meter
count,"14,917.00"
mean,59.40
std,29.85
min,14.50
25%,41.10
50%,52.54
75%,68.00
max,456.00


### Number of Listings by Size, separated by Primary and Secondary Market

In [ ]:
fig = px.histogram(data_with_price, x='Square_meter',
                   color='Market',
                   hover_data=data.columns,
                   template='plotly_white',
                   range_y=(0, 750),
                   range_x=(0,250),
                   color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                       'pierwotny':px.colors.sequential.haline[10]},
                   labels={'Market':'Rynek',
                           'Square_meter':'Metraż'})
fig.update_layout(
                  xaxis_title='<b>Metraż nieruchomości</b>',
                  yaxis_title='<b>Liczba ogłoszeń</b>',)
# title='<b>Ilość ogłoszeń w zależnośći od metrażu</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>',
fig.show()
fig.write_html('hist_liczba_metraz.html')

### Price Distribution, separated by Primary and Secondary Market

###Price Distribution, separated by District, Primary and Secondary Market

In [ ]:
fig = px.box(data_with_price,
             x='District',
             y='Price_sq_meter',
             color='Market',
             template='plotly_white',
             range_y=(0, 50500),
             color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                 'pierwotny':px.colors.sequential.haline[10]},
             labels={'Market':'Rynek',
                     'District':'Dzielnica',
                     'Price_sq_meter':'Cena za m<sup>2</sup>'})
fig.update_layout(xaxis_title='<b>Dzielnica</b>',
                  yaxis_title='<b>Cena za m<sup>2</sup></b>')
# title='<b>Rozkład cen</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>',
# fig.update_yaxes(type='log',)
fig.show()
fig.write_html('box_cena_m2_dzielnica.html')

In [ ]:
fig = px.box(data_with_price,
             x='District',
             y='Price_PLN',
             color='Market',
             template='plotly_white',
             # range_y=(0, 50500),
             color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                 'pierwotny':px.colors.sequential.haline[10]},
             labels={'Market':'Rynek',
                     'District':'Dzielnica',
                     'Price_sq_meter':'Cena nieruchomości'})
fig.update_layout(xaxis_title='<b>Dzielnica</b>',
                  yaxis_title='<b>Cena nieruchomości')
# title='<b>Rozkład cen</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>',
fig.update_yaxes(type='log',)
fig.show()
fig.write_html('box_cena_dzielnica.html')

#Correlation Between Area, Number of Rooms, and Price

In [ ]:
fig = px.scatter(data_with_price,
                 x='Price_PLN',
                 y='Price_sq_meter',
                 color='Market',
                 template='plotly_white',
                 color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                 'pierwotny':px.colors.sequential.haline[10]},
                 labels={'Market':'Rynek'})
fig.update_layout(xaxis_title='<b>Cena nieruchomości</b>',
                  yaxis_title='<b>Cena za m<sup>2</sup></b>',
                  # title='<b>Zależność między ceną, a ceną za m2</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>'
                  )
fig.update_xaxes(type='log',)
fig.show()

In [ ]:
fig = px.scatter(data_with_price,
                 x='Price_PLN',
                 y='Square_meter',
                 color='Market',
                 template='plotly_white',
                 color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                 'pierwotny':px.colors.sequential.haline[10]},
                 labels={'Market':'Rynek'})
fig.update_layout(xaxis_title='<b>Cena nieruchomości</b>',
                  yaxis_title='<b>Metraż nieruchomości</b>',
                  # title='<b>Zależność między ceną, a powierzchnią nieruchomości</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>'
                  )
fig.update_xaxes(type='log',)
fig.show()

In [ ]:
fig = px.scatter(data_with_price,
                 x='Price_sq_meter',
                 y='Square_meter',
                 size='Rooms',
                 size_max=25,
                 color='Market',
                 template='plotly_white',
                 color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                 'pierwotny':px.colors.sequential.haline[10]},
                 labels={'Market':'Rynek'})
fig.update_layout(xaxis_title='<b>Cena nieruchomości</b>',
                  yaxis_title='<b>Metraż nieruchomości</b>',
                  # title='<b>Zależność między ceną, powierzchnią nieruchomości oraz ilością pokoi</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>'
                  )
fig.update_xaxes(type='log',)
fig.show()

#Geographical Analysis of Prices and Listings

##Average Price per Apartment by District

In [ ]:
price_by_district = data_with_price.groupby(['District', 'Market'], as_index=False).agg({'Price_PLN': pd.Series.mean}).sort_values("Price_PLN", ascending=False)
price_sqm_by_district = data_with_price.groupby(['District', 'Market'], as_index=False).agg({'Price_sq_meter': pd.Series.mean}).sort_values("Price_sq_meter", ascending=False)
price_by_district_median = data_with_price.groupby(['District', 'Market'], as_index=False).agg({'Price_PLN': pd.Series.median}).sort_values("Price_PLN", ascending=False)
price_sqm_by_district_median = data_with_price.groupby(['District', 'Market'], as_index=False).agg({'Price_sq_meter': pd.Series.median}).sort_values("Price_sq_meter", ascending=False)

### Average Price per Property - Primary Market

In [ ]:
fig = px.bar(price_by_district[price_by_district['Market'] == 'pierwotny'],
             x='District',
             y='Price_PLN',
             text_auto='.3s',
             template='plotly_white',
             labels={'District':'Dzielnica', 'Price_PLN':'Cena'})
fig.update_layout(xaxis_title='<b>Dzielnica</b>',
                  yaxis_title='<b>Średnia cena nieruchomości</b>')
fig.update_traces(marker_color=px.colors.sequential.haline[10],
                  textposition="outside",
                  cliponaxis=False)
# title='<b>Średnia cena za nieruchomość</b><br><sup>rynek pierwotny</sup>'

fig.show()
fig.write_html('bar_pierwotny_cena_dzielnica.html')

### Average Price per Property - Secondary Market

In [ ]:
fig = px.bar(price_by_district[price_by_district['Market'] == 'wtórny'],
             x='District',y='Price_PLN',
             text_auto='.3s',
             template='plotly_white',
             color_discrete_sequence=px.colors.sequential.haline,
             labels={'District':'Dzielnica', 'Price_PLN':'Cena'})
fig.update_layout(xaxis_title='<b>Dzielnica</b>',
                  yaxis_title='<b>Średnia cena nieruchomości</b>')
# title='<b>Średnia cena za nieruchomość</b><br><sup>rynek wtórny</sup>'
fig.update_traces(textposition="outside", cliponaxis=False)

fig.show()
fig.write_html('bar_wtorny_cena_dzielnica.html')

### Average Price per m² - Primary Market

In [ ]:
fig = px.bar(price_sqm_by_district[price_sqm_by_district['Market'] == 'pierwotny'],
             x='District',
             y='Price_sq_meter',
             text_auto='.4s',
             template='plotly_white',
             color_discrete_sequence=px.colors.sequential.haline,
             labels={'District':'Dzielnica', 'Price_sq_meter':'Cena'},
             range_y=(0, 31000))
fig.update_layout(xaxis_title='<b>Dzielnica</b>',
                  yaxis_title='<b>Średnia cena za m<sup>2</sup></b>')
# title='<b>Średnia cena za m<sup>2</sup></b><br><sup>rynek pierwotny</sup>'
fig.update_traces(marker_color=px.colors.sequential.haline[10],
                  textposition="outside",
                  cliponaxis=False)

fig.show()
fig.write_html('bar_pierwotny_cena_m2_dzielnica.html')

### Average Price per m² - Secondary Market

In [ ]:
fig = px.bar(price_sqm_by_district[price_sqm_by_district['Market'] == 'wtórny'],
             x='District',
             y='Price_sq_meter',
             text_auto='.4s',
             #text=price_by_district['Price_sq_meter'],
             #text=data['Price_PLN'].apply(lambda x: f'{x/1000000:.1f}'),
             template='plotly_white',
             color_discrete_sequence=px.colors.sequential.haline,
             labels={'District':'Dzielnica', 'Price_sq_meter':'Cena'},
             range_y=(0, 31000))
fig.update_layout(xaxis_title='<b>Dzielnica</b>',
                  yaxis_title='<b>Średnia cena za m<sup>2</sup></b>')
# title='<b>Średnia cena za m<sup>2</sup></b><br><sup>rynek wtórny</sup>'
fig.update_traces(textposition="outside", cliponaxis=False)

fig.show()
fig.write_html('bar_wtorny_cena_m2_dzielnica.html')

#District Map

###Average Price per Apartment by District - Primary Market

In [ ]:
with open("warszawa-dzielnice.geojson", "r", encoding="utf-8") as file:
    warsaw_geojson = json.load(file)

fig = px.choropleth_mapbox(
    price_sqm_by_district[price_sqm_by_district.Market == 'pierwotny'],
    geojson=warsaw_geojson,
    locations='District',
    featureidkey='properties.name', # ścieżka do nazwy dzielnicy w pliku GeoJSON
    color='Price_sq_meter',
    center={"lat": 52.2319, "lon": 21.0067},
    mapbox_style="carto-positron",
    # mapbox_style="open-street-map",
    zoom=9,
    color_continuous_scale='haline',
    opacity=0.7,
    range_color=(price_sqm_by_district[price_sqm_by_district.Market == 'pierwotny'].Price_sq_meter.min(),
                 price_sqm_by_district[price_sqm_by_district.Market == 'pierwotny'].Price_sq_meter.max()),
    labels={'Price_sq_meter':'Średnia cena za m<sup>2</sup>', 'District':'Dzielnica'},
    color_continuous_midpoint=20000
    )

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
# title='<b>Średnia cena za nieruchomość z podziałem na dzielnice</b><br><sup>rynek pierwotny</sup>',
fig.show()
fig.write_html('map_cena_pierwotny.html')

###Average Price per Apartment by District - Secondary Market

In [ ]:
with open("warszawa-dzielnice.geojson", "r", encoding="utf-8") as file:
    warsaw_geojson = json.load(file)

fig = px.choropleth_mapbox(
    price_sqm_by_district[price_sqm_by_district.Market == 'wtórny'],
    geojson=warsaw_geojson,
    locations='District',
    featureidkey='properties.name', # ścieżka do nazwy dzielnicy w pliku GeoJSON
    color='Price_sq_meter',
    center={"lat": 52.2319, "lon": 21.0067},
    mapbox_style="carto-positron",
    # mapbox_style="open-street-map",
    zoom=9,
    color_continuous_scale='haline',
    opacity=0.7,
    range_color=(price_sqm_by_district[price_sqm_by_district.Market == 'wtórny'].Price_sq_meter.min(),
                 price_sqm_by_district[price_sqm_by_district.Market == 'wtórny'].Price_sq_meter.max()),
    labels={'Price_sq_meter':'Średnia cena za m<sup>2</sup>', 'District':'Dzielnica'},
    color_continuous_midpoint=20000
    )
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
# title='<b>Średnia cena za nieruchomość z podziałem na dzielnice</b><br><sup>rynek wtórny</sup>'

fig.show()
fig.write_html('map_cena_wtorny.html')

#Time-Dependent Analysis

In [ ]:
data_available_from = data.dropna(subset=['Available_from'])
now = datetime.datetime.now()

In [ ]:
data_available_from = data_available_from[data_available_from.Available_from >= now]

###Availability of Apartments Over Time, separated by Primary and Secondary Market

In [ ]:
fig = px.histogram(data_available_from,
                   x='Available_from',
                   color='Market',
                   template="plotly_white",
                   color_discrete_map={'wtórny':px.colors.sequential.haline[0],
                                       'pierwotny':px.colors.sequential.haline[10]},
                   labels={'Market':'Rynek',
                           'Available_from':'Dostępne od',
                           'count':'ilość'})
fig.update_layout(xaxis_title='',
                  yaxis_title='<b>Ilość ogłoszeń</b>',)
# title='<b>Dostępność czasowa nieruchomości</b><br><sup>z podziałem na rynek pierwotny/wtórny</sup>'

fig.show()
fig.write_html('hist_dostepnosc.html')